# DS207 Final Project - Predicting Readmission Rate for Diabetic Patients Using Machine Learning
#### Contributor: Colin Frishberg (cpfrish@berkeley.edu)

### Notebook Structure:

1.  Create compelling **data visualizations**.
2.  **Input:** Load raw data from data file.
3.  Build, train, and evaluate the **Transformer Model**.
4.  **Output:** Save the trained Transformer model to `/results/transformer.h5`.
5.  **Output:** Save the statistics from the transformer model to `/results/stats_transformer.csv`.

## 1. Notebook Imports

### 1.1 Import basic and necessary libraries:

In [ ]:
# Basic imports
import numpy as np
import warnings
import pandas as pd
from matplotlib import pyplot as plt
import tensorflow as tf
from tensorflow import keras
import altair as alt
alt.data_transformers.enable('vegafusion')

# Compress all warnings
warnings.filterwarnings("ignore")

### 1.2 Import datasets:

In [ ]:
# Training dataset
df_train = pd.read_csv('../results/train.csv')
X_train = df_train.drop(columns=['readmitted'], axis=1)
Y_train = df_train['readmitted']
# Validation dataset
df_val = pd.read_csv('../results/val.csv')
X_val = df_val.drop(columns=['readmitted'], axis=1)
Y_val = df_val['readmitted']
# Testing dataset
df_test = pd.read_csv('../results/test.csv')
X_test = df_test.drop(columns=['readmitted'], axis=1)
Y_test = df_test['readmitted']

In [ ]:
# Display top 5 rows of the training dataset
display(df_train.head())

In [ ]:
# Check the shape of all variables
print(f"The shape of X_train is: {X_train.shape}")
print(f"The shape of Y_train is: {Y_train.shape}")
print(f"The shape of X_val is: {X_val.shape}")
print(f"The shape of Y_val is: {Y_val.shape}")
print(f"The shape of X_test is: {X_test.shape}")
print(f"The shape of Y_test is: {Y_test.shape}")

## 2. Visualization

Visualization requirements: Include multiple detailed plots that effectively communicate your data insights: ensure all plots have properly labeled x and y axes; include descriptive titles; add legends where appropriate; consider using multiple plot types (histograms, scatter plots, box plots, heatmaps, etc.) to highlight different aspects of your data; accompany each visualization with interpretations of what the patterns reveal.

In [ ]:
# Install vegafusion for Altair visualizations
%pip install "vegafusion[embed]>=1.5.0"
# Install vl-convert-python for Altair visualizations
%pip install "vl-convert-python>=1.6.0"

#### Kernel Density Estimates

In [ ]:
continuous_features = [ 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 
                       'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'age_mid']

In [ ]:
# Kernel Density Estimates for continuous features

kde_vars = continuous_features

def create_kde_plots(df, kde_vars, ncols=2):
  """
  Generates and combines KDE plots into a grid for given features

  Args:
      df (pd.Dataframe)
      kde_vars(list): List of columns to plot
      ncols(int): Number of columns in the grid

  Returns:
      alt.vconcat: Grid of KDE plots
  """
  charts = []
  for var in kde_vars:
    chart = alt.Chart(df).transform_density(
        density=var,
        as_=[var, 'Density'], # The output fields for the value and its density
        groupby=['readmitted']
    ).mark_area(opacity=0.5).encode(
        x=alt.X(f'{var}:Q', title=var.replace('_', ' ').title()),
        y=alt.Y('Density:Q'),
        # Color by the 'readmitted' class
        color=alt.Color('readmitted:N', title='Readmitted Class')
    ).properties(
        title=f'Distribution of {var.replace("_", " ").title()}',
        width=300,
        height=200
    )
    charts.append(chart)

    # Grid creations
    rows = [
       alt.hconcat(*charts[i:i+ncols])
       for i in range(0, len(charts), ncols)
    ]
  # Combine all created charts vertically and return them
  return alt.vconcat(*rows)


In [ ]:
kde_plots = create_kde_plots(viz_df, kde_vars, ncols=4)
kde_plots


#### Box Plots

In [ ]:
# Box Plots for distributions of key predictors to outcome

box_plots = []
for feature in continuous_features:
    # Create box plot for each feature against readmission
    chart = alt.Chart(viz_df).mark_boxplot().encode(
        x=alt.X('readmitted:N', title='Readmitted Class (0: NO, 1: >30, 2: <30)'),
        y=alt.Y(f'{feature}:Q', title=feature.replace('_', ' ').title()),
        color=alt.Color('readmitted:N', title='Readmitted Class').scale(scheme='category20'),
        tooltip=['readmitted', feature]
    ).properties(
        title=f'Box Plot of {feature.replace("_", " ").title()} by Readmission Status',
        width=300,
        height=200
    )
    box_plots.append(chart)

# Arrange the box plots in a grid
ncols = 3
rows = [alt.hconcat(*box_plots[i:i+ncols]) for i in range(0, len(box_plots), ncols)]
alt.vconcat(*rows)

#### Violin Plots

In [ ]:
df = pd.concat([df_train, df_val, df_test], axis=0)

In [ ]:
# Violin Plots
continuous_features = [ 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 
                       'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

violin_plots = []
for feature in continuous_features:
    chart = alt.Chart(df).transform_density(
        density=feature,
        as_=[feature, 'density'],
        groupby=['readmitted']
    ).mark_area(orient='horizontal').encode(
        y=alt.Y(f'{feature}:Q', title=feature.replace('_', ' ').title()),
        x=alt.X(
            'density:Q',
            stack='center',
            impute=None,
            title=None,
            axis=alt.Axis(labels=False, values=[0],grid=False, ticks=True),
        ),
        color=alt.Color('readmitted:N', legend=alt.Legend(title="Readmitted")).scale(scheme='tableau20'),
    ).properties(
        title=f'Distribution of {feature.replace("_", " ").title()} by Readmission',
        width=300,
        height=250
    )
    violin_plots.append(chart)

# Arrange the violin plots in a grid
ncols = 4
rows_violin = [alt.hconcat(*violin_plots[i:i+ncols]) for i in range(0, len(violin_plots), ncols)]
final_violin_chart = alt.vconcat(*rows_violin)


final_violin_chart

## 3. Model Developments

In [ ]:
def build_transformer_model(
    input_shape: int,
    num_classes: int = 2,
    embedding_dim: int = 192,
    num_heads: int = 8,
    key_dim: int = 24,
    ffn_dim: int = 2048,
    dense_dim: int = 64,
) -> keras.Model:
    """
    Build a transformer model for readmission prediction.

    Args:
        input_shape: Number of input features
        num_classes: Number of output classes
        embedding_dim: Dimension of embedding layer
        num_heads: Number of attention heads
        key_dim: Dimension of key in attention
        ffn_dim: Dimension of feed-forward network
        dense_dim: Dimension of final dense layer

    Returns:
        Compiled Keras model
    """
    # # Use defaults from config if not provided
    # if embedding_dim is None:
    #     embedding_dim = config.TRANSFORMER_PARAMS["embedding_dim"]
    # if num_heads is None:
    #     num_heads = config.TRANSFORMER_PARAMS["num_heads"]
    # if key_dim is None:
    #     key_dim = config.TRANSFORMER_PARAMS["key_dim"]
    # if ffn_dim is None:
    #     ffn_dim = config.TRANSFORMER_PARAMS["ffn_dim"]
    # if dense_dim is None:
    #     dense_dim = config.TRANSFORMER_PARAMS["dense_dim"]

    inputs = keras.Input(shape=(input_shape,))

    # Embedding layer
    x = keras.layers.Dense(embedding_dim, activation="relu", kernel_regularizer=keras.regularizers.l2(0.01))(inputs)
    x = keras.layers.Reshape((1, embedding_dim))(x)

    #Dropout layer
    x = keras.layers.Dropout(0.1)(x)

    # Transformer block
    attention_output = keras.layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim
    )(x, x)
    x = keras.layers.Add()([x, attention_output])
    x = keras.layers.LayerNormalization()(x)

    # Feed-forward network
    ffn_output = keras.layers.Dense(ffn_dim, activation="relu", kernel_regularizer=keras.regularizers.l2(0.01))(x)
    ffn_output = keras.layers.Dense(embedding_dim, kernel_regularizer=keras.regularizers.l2(0.01))(ffn_output)
    x = keras.layers.Add()([x, ffn_output])
    x = keras.layers.LayerNormalization()(x)

    # Pooling layer
    x = keras.layers.GlobalAveragePooling1D()(x)

    # Dropout layer
    x = keras.layers.Dropout(0.1)(x)

    # Output layers
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(dense_dim, activation="relu", kernel_regularizer=keras.regularizers.l2(0.01))(x)
    outputs = keras.layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

In [ ]:
def train_transformer_model(
    X_train: pd.DataFrame,
    Y_train: pd.Series,
    X_val: pd.DataFrame,
    Y_val: pd.Series,
    epochs: int = 10,
    batch_size: int = 64,
    verbose: int = 1,
) -> Tuple[keras.Model, keras.callbacks.History]:
    """
    Train a transformer model for readmission prediction.

    Args:
        X_train: Training features
        Y_train: Training target
        X_val: Validation features
        Y_val: Validation target
        epochs: Number of training epochs
        batch_size: Batch size for training
        verbose: Verbosity level (0, 1, or 2)

    Returns:
        Tuple of (trained model, training history)
    """

    early_stopping_callback = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        min_delta=0.001,
        patience=5,
        verbose=1,
        mode ="min",
        restore_best_weights=True,      
    )   

    input_shape = X_train.shape[1]
    num_classes = len(np.unique(Y_train))

    model = build_transformer_model(input_shape, num_classes)

    # Compile model
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    # Train model
    history = model.fit(
        X_train,
        Y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_val, Y_val),
        callbacks=[early_stopping_callback],
        verbose=verbose,
    )

    return model, history

In [ ]:
model_tran, history = train_transformer_model(
    X_train=X_train,
    Y_train=Y_train,
    X_val=X_val,
    Y_val=Y_val,
    epochs=10,
    batch_size=64,
    verbose=1,
)

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot accuracy
ax1.plot(history.history["accuracy"], label="Train Accuracy")
ax1.plot(history.history["val_accuracy"], label="Val Accuracy")
ax1.set_title("Model Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(alpha=0.3)

# Plot loss
ax2.plot(history.history["loss"], label="Train Loss")
ax2.plot(history.history["val_loss"], label="Val Loss")
ax2.set_title("Model Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Notebook Exports

#### 3.1 Export models

In [ ]:
# Export the transformer model into results/
model_tran.save('../results/transformer.h5')

#### 3.2 Export evaluation results

In [ ]:
# # Initialize the dataframe
# stats = pd.DataFrame(columns=['feature_importance', 'precision', 'recall',
#                               'false_positive', 'false_negative', 
#                               'final_accuracy_train', 'final_accuracy_test', 'f1'])

# # Insert model statistics from the transformer model
# stats_tran = pd.DataFrame([{'feature_importance': ,
#                           'precision': ,
#                           'recall': ,
#                           'false_positive' : ,
#                           'false_negative' : ,
#                           'final_accuracy_train' : ,
#                           'final_accuracy_test' : ,
#                           'f1' : }])

# # Combine the stats and export as csv
# stats = pd.concat([stats, stats_tran], ignore_index=True)
# stats.to_csv('../results/stats_transformer.csv', index=False) 